Task 4 : Sentiment Analysis

In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
#pandas → used to load and handle dataset
#numpy → used for numerical operations
#re → used for text cleaning (removing symbols, etc.)
#nltk → Natural Language Processing library

#Stopwords = common words like “the”, “is”, “and”
#These words don’t add meaning in sentiment analysis
#We remove them to improve model performance

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
#Load Dataset
df = pd.read_csv("model_output.csv", engine='python', on_bad_lines='skip')
df.head()
#Loads dataset into a DataFrame
#head() shows first 5 rows
#Helps understand structure of data

,rating,review_title,review_content,Sentiment
0,3.8,"This is a Best kodak LED,Product is Good as pe...",",Product is Good as per the price but customer...",-1
1,4.2,Cover is perfect size wise and it's exactly sa...,Cover is perfect size wise and it's exactly sa...,1
2,4.1,"Awesome Product,Good product,Good quality,Good...","It is good for data cables...I liked it,https:...",1
3,4.3,"7-8/10, Decent, good for day to day use,Good c...","2 months review- its been working fine, there'...",1
4,4.2,Cheap product and same is the performance but ...,The signal is too unstable once connect you ne...,1


In [ ]:
#Select Required Columns
df = df[['rating', 'review_content']]
#We only need:
#Score → rating (target)
#Text → review (input)
#Removes unnecessary columns

#Remove Missing Values
df = df.dropna()
#Removes rows with empty values
#Ensures clean and usable data

In [ ]:
#Text Cleaning
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
#Loads list of stopwords
#Converts to set for faster checking
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
df['Cleaned_Text'] = df['review_content'].apply(clean_text)

text.lower()
→ converts all text to lowercase
re.sub(r'[^a-zA-Z]', ' ', text)
→ removes numbers, punctuation, special characters
text.split()
→ splits sentence into words
[w for w in words if w not in stop_words]
→ removes useless words like “is”, “the”
" ".join(words)
→ joins words back into sentence


Applies cleaning function to every review
Stores result in new column Cleaned_Text


In [ ]:
#Create Sentiment Labels
def sentiment(score):
    if score >= 4:
        return "Positive"
    elif score == 3:
        return "Neutral"
    else:
        return "Negative"

#Converts numeric rating into sentiment:
#4–5 → Positive
#3 → Neutral
#1–2 → Negative

df['Sentiment'] = df['rating'].apply(sentiment)
#Applies function to each row
#Creates new column Sentiment
# Check class distribution
print(df['Sentiment'].value_counts())
from sklearn.utils import resample

# Separate classes
df_pos = df[df['Sentiment'] == "Positive"]
df_neg = df[df['Sentiment'] == "Negative"]
df_neu = df[df['Sentiment'] == "Neutral"]

# Downsample positive class
df_pos_down = resample(df_pos,
                      replace=False,
                      n_samples=len(df_neg),
                      random_state=42)

# Combine balanced dataset
df_balanced = pd.concat([df_pos_down, df_neg, df_neu])

Sentiment
Positive    509
Negative    160
Name: count, dtype: int64


In [ ]:
#Convert Text to Numbers (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df_balanced['Cleaned_Text'])
y = df_balanced['Sentiment']
#TF-IDF converts text → numerical form
#max_features=5000 → limits vocabulary size
#X → input features (text converted to numbers)
#y → output labels (Positive/Neutral/Negative)

In [ ]:
#Split Dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# Train Model
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)
#Uses Naive Bayes algorithm
#Good for text classification
#fit() trains model using training data

#Prediction
y_pred = model.predict(X_test)
#Predicts sentiment for test data

In [ ]:
#Evaluate Model
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
#accuracy_score → % of correct predictions
#classification_report → precision, recall, F1-score

Accuracy: 0.65625
              precision    recall  f1-score   support

    Negative       0.64      0.72      0.68        32
    Positive       0.68      0.59      0.63        32

    accuracy                           0.66        64
   macro avg       0.66      0.66      0.65        64
weighted avg       0.66      0.66      0.65        64



In [ ]:
df.to_csv("final_sentiment_data.csv", index=False)

In [ ]:
#example
sample = ["This product is bad"]
sample_clean = [clean_text(sample[0])]
sample_vec = tfidf.transform(sample_clean)

prediction = model.predict(sample_vec)
print(prediction)

['Negative']
